# 01 — Preprocessing podatkov (HNSCC, GSE200996)

Iz surovih GEO podatkov naredimo `.pkl` fajle ki jih TRIM potrebuje:
- `data_rna.pkl` — RNA ekspresija (celice × geni)
- `data_labels.pkl` — oznake vsake celice (tkivo, čas, pacient, TCR indeks)
- `data_labels_str.pkl` — iste oznake kot besedilo
- `df_all_tcrs.pkl` — seznam vseh unikatnih TCR sekvenc

## 0. Namestitev knjižnic in nastavitev poti

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
# Namesti manjkajoče knjižnice (samo enkrat)
!pip install -q --upgrade scanpy scipy

: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import pickle
import glob

# Poti — prilagodi če imaš drugačno strukturo na Drive
RAW_DIR    = '/content/drive/MyDrive/Diploma/data/GSE200996_RAW'
META_DIR   = '/content/drive/MyDrive/Diploma/data'
OUTPUT_DIR = '/content/drive/MyDrive/Diploma/data/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Poti nastavljene.')

## 1. Naloži metadata fajle

Vsak barcode (ID celice) ima pripisane: pacienta, fazo (B1/B2/Pre-Tx/Post-Tx), tip celice.

In [ ]:
def load_meta(filepath):
    df = pd.read_csv(filepath, sep='\t', index_col=0)
    return df

# PBMC (kri) — CD4 in CD8
meta_pbmc_cd4 = load_meta(os.path.join(META_DIR, 'GSE200996_CD4.PBMC.single.cell.meta.data.txt'))
meta_pbmc_cd8 = load_meta(os.path.join(META_DIR, 'GSE200996_CD8.PBMC.single.cell.meta.data.txt'))

# Tumor — CD4 in CD8
meta_tumor_cd4 = load_meta(os.path.join(META_DIR, 'GSE200996_CD4.tumor.single.cell.meta.data.txt'))
meta_tumor_cd8 = load_meta(os.path.join(META_DIR, 'GSE200996_CD8.tumor.single.cell.meta.data.txt'))

# Dodaj oznake
meta_pbmc_cd4['CellClass'] = 'CD4';  meta_pbmc_cd4['Tissue_str'] = 'Blood'
meta_pbmc_cd8['CellClass'] = 'CD8';  meta_pbmc_cd8['Tissue_str'] = 'Blood'
meta_tumor_cd4['CellClass'] = 'CD4'; meta_tumor_cd4['Tissue_str'] = 'Tumor'
meta_tumor_cd8['CellClass'] = 'CD8'; meta_tumor_cd8['Tissue_str'] = 'Tumor'

meta_all = pd.concat([meta_pbmc_cd4, meta_pbmc_cd8, meta_tumor_cd4, meta_tumor_cd8])

# Izpusti B3 — nimajo scRNA-seq (.h5) fajlov
meta_all = meta_all[meta_all['Stage'] != 'B3']

print(f'Skupaj celic v metadata (brez B3): {len(meta_all)}')
print(meta_all['Stage'].value_counts())

## 2. Naloži RNA `.h5` fajle

Vsak `.h5` fajl vsebuje count matriko (surovo število branj na gen na celico) za skupino pacientov.
Barcode vsake celice je ključ s katerim povežemo RNA z metadata.

In [ ]:
import anndata
import re

def get_patients_from_filename(filename):
    """Iz imena fajla izvleče seznam pacientov, npr. P01-P02 → ['P01', 'P02']"""
    basename = os.path.basename(filename)
    match = re.search(r'(P\d{2}(?:-P\d{2})*)', basename)
    if match:
        return match.group(1).split('-')
    return None

def load_h5_files(pattern, meta_df, min_genes=200, max_mito_pct=20):
    """
    Naloži .h5 fajle. Za vsak fajl iz imena razberemo kateri pacienti so noter,
    nato filtriramo samo barcodes ki pripadajo tem pacientom v metadata.
    Shranimo barcode_clean in Patient_ID za vsako celico.

    QC (2 koraka, standardni scRNA — Luecken & Theis 2019):
    (1) min_genes=200: GEO ponuja SAMO raw_feature_bc_matrix (~737k barkod, vecina
        empty droplets). Distribucija genov je BIMODALNA (dropleti ~2-10 countov vs
        prave celice 1000+; siva cona 50-200 skoraj prazna). Prag 200 odstrani sum,
        obdrzi prave celice. Empiricno: dvignilo ekspanzija AUC 0.52 -> 0.77.
    (2) mitohondrijski % < 20: umirajoce/razpadle celice (iztekla citoplazma) imajo
        nesorazmerno visok delez mitohondrijskih transkriptov (MT- geni). Odstranimo
        celice z mito% >= max_mito_pct kot low-quality. Diagnostika izpise koliko jih pade.
    """
    files = sorted(glob.glob(pattern))
    print(f'Najdenih {len(files)} fajlov za vzorec: {os.path.basename(pattern)}')
    adatas = []
    for f in files:
        patients = get_patients_from_filename(f)
        if patients is None:
            print(f'  OPOZORILO: ne morem razbrali pacientov iz {os.path.basename(f)}, preskočim')
            continue

        # Barcodes iz metadata za te paciente: barcode_clean → Patient_ID lookup
        mask_patients = meta_df['Patient_ID'].isin(patients)
        meta_sub = meta_df[mask_patients].copy()
        meta_sub['barcode_clean'] = meta_sub.index.str.split('_').str[0]
        # Composer: barcode_clean → Patient_ID (unikatno znotraj fajla)
        bc_to_patient = meta_sub.groupby('barcode_clean')['Patient_ID'].first()
        valid_bc = set(bc_to_patient.index)

        adata = sc.read_10x_h5(f)
        adata.var_names_make_unique()

        # QC (1): odstrani empty droplets/sum (celice z <min_genes izrazenih genov).
        n0 = adata.shape[0]
        sc.pp.filter_cells(adata, min_genes=min_genes)
        n1 = adata.shape[0]

        bc_clean = adata.obs_names.str.replace(r'-\d+$', '', regex=True)
        mask = bc_clean.isin(valid_bc)
        adata = adata[mask].copy()
        bc_clean_filtered = bc_clean[mask].values
        adata.obs['barcode_clean'] = bc_clean_filtered
        adata.obs['Patient_ID'] = [bc_to_patient[bc] for bc in bc_clean_filtered]

        # QC (2): mitohondrijski % filter (MT- geni). Racunaj na avtorjevih (metadata) celicah.
        adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
        sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
        n2 = adata.shape[0]
        n_high_mito = int((adata.obs['pct_counts_mt'] >= max_mito_pct).sum())
        adata = adata[adata.obs['pct_counts_mt'] < max_mito_pct].copy()

        adatas.append(adata)
        print(f'  {os.path.basename(f)} (P{patients}): raw {n0} -> genov>={min_genes} {n1} -> po barcode {n2} -> mito%<{max_mito_pct} {adata.shape[0]}  (odstranjenih zaradi mito: {n_high_mito})')

    adatas = [a for a in adatas if a.shape[0] > 0]
    combined = anndata.concat(adatas, join='inner', index_unique=None)
    return combined

print('Funkcija pripravljena (QC: min_genes=200 + mito%<20 + avtorjev metadata match).')

## 2b. Pregled dropletov PO RAZPONU countov (pred QC filtracijo)

Diagnostika, ki UTEMELJI prag `min_genes=200`. Naloži surove `.h5` (z metadata-matchingom,
a **BREZ** `filter_cells`) in pokaze, koliko celic pade v vsak razpon countov/genov.

Hipoteza: bimodalno — problematicne kaplje (dropleti/ambientni sum) imajo ~malo countov
(~2-10), prave T-celice mnogo vec (1000+); siva cona (50-200) skoraj prazna → prag 200 varen.
Poglej graf: rdeci stolpci (pod 200) so sum, ki ga QC odstrani.

In [ ]:
import numpy as np, gc
import matplotlib.pyplot as plt

# Naloži surove .h5 (metadata match, BREZ filter_cells) samo za PREGLED distribucije.
def load_h5_raw_for_qc(pattern, meta_df):
    adatas = []
    for f in sorted(glob.glob(pattern)):
        patients = get_patients_from_filename(f)
        if patients is None: continue
        meta_sub = meta_df[meta_df['Patient_ID'].isin(patients)].copy()
        meta_sub['barcode_clean'] = meta_sub.index.str.split('_').str[0]
        valid_bc = set(meta_sub['barcode_clean'])
        ad = sc.read_10x_h5(f); ad.var_names_make_unique()
        bc = ad.obs_names.str.replace(r'-\d+$', '', regex=True)
        ad = ad[bc.isin(valid_bc)].copy()
        adatas.append(ad)
    adatas = [a for a in adatas if a.shape[0] > 0]
    return anndata.concat(adatas, join='inner', index_unique=None)

print('Nalagam surove .h5 za QC pregled (brez filtriranja)...')
_qc = anndata.concat([
    load_h5_raw_for_qc(os.path.join(RAW_DIR, '*_GEX_sc_PBMC.h5'),  meta_all),
    load_h5_raw_for_qc(os.path.join(RAW_DIR, '*_GEX_sc_tumor.h5'), meta_all)
], join='inner', index_unique=None)

counts_per_cell = np.asarray(_qc.X.sum(axis=1)).ravel()
genes_per_cell  = np.asarray((_qc.X > 0).sum(axis=1)).ravel()
THRESH = 200

edges  = [0, 10, 50, 100, 200, 500, 1000, 2000, 5000, np.inf]
labels = ['0-10', '10-50', '50-100', '100-200', '200-500', '500-1k', '1k-2k', '2k-5k', '5k+']
def bin_counts(d):
    idx = np.digitize(d, edges[1:-1], right=False)
    return np.array([(idx == i).sum() for i in range(len(labels))])
n_counts, n_genes = bin_counts(counts_per_cell), bin_counts(genes_per_cell)

print('Skupaj celic (metadata match, PRED QC):', len(counts_per_cell))
print()
hdr = '{:>10} | {:>20} | {:>20}'.format('razpon', 'UMI countov', 'izrazenih genov')
print(hdr); print('-'*56)
for lab, nc, ng in zip(labels, n_counts, n_genes):
    mark = '  <- prag 200' if lab == '100-200' else ''
    line = '{:>10} | {:>8} ({:>4.1f}%) | {:>8} ({:>4.1f}%){}'.format(
        lab, nc, 100*nc/len(counts_per_cell), ng, 100*ng/len(genes_per_cell), mark)
    print(line)

# graf razponov
INK, ACCENT, GRID = '#1a1a2e', '#e4572e', '#e8e8ec'
x = np.arange(len(labels)); below = [i for i,l in enumerate(labels) if l in ['0-10','10-50','50-100','100-200']]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, vals, name in [(axes[0], n_counts, 'UMI countov'), (axes[1], n_genes, 'izrazenih genov')]:
    ax.bar(x, vals, color=[ACCENT if i in below else INK for i in x], width=0.72)
    for xi, v in zip(x, vals):
        if v > 0: ax.text(xi, v, str(v), ha='center', va='bottom', fontsize=8, color=INK)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_xlabel(name + ' na celico'); ax.set_title('St. celic po razponu: ' + name, color=INK)
    ax.grid(axis='y', color=GRID, lw=0.8)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)
axes[0].set_ylabel('st. celic')
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=ACCENT, label='pod 200 (sum, QC odstrani)'),
                        Patch(color=INK, label='nad 200 (prave celice)')], fontsize=8, loc='upper right')
fig.suptitle('Droplet distribucija PRED QC (utemeljitev min_genes=200)', fontweight='bold')
fig.tight_layout(); plt.show()

gray = np.logical_and(genes_per_cell >= 50, genes_per_cell < 200)
print()
print('Pod pragom 200 genov: {} ({:.1f}%)'.format(int((genes_per_cell<200).sum()), 100*(genes_per_cell<200).mean()))
verdikt = 'BIMODALNO, prag 200 varen' if gray.mean() < 0.05 else 'siva cona ni prazna'
print('Siva cona (50-199): {} ({:.1f}%)  -> {}'.format(int(gray.sum()), 100*gray.mean(), verdikt))

# --- MITOHONDRIJSKI % diagnostika (pred mito filtrom) ---
_qc.var['mt'] = _qc.var_names.str.upper().str.startswith('MT-')
n_mt = int(_qc.var['mt'].sum())
print()
print('MT- (mitohondrijskih) genov najdenih:', n_mt)
if n_mt > 0:
    sc.pp.calculate_qc_metrics(_qc, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    mp = _qc.obs['pct_counts_mt']
    print('  mito% distribucija: mediana {:.1f}%, 90-perc {:.1f}%, max {:.1f}%'.format(
        mp.median(), mp.quantile(0.9), mp.max()))
    for th in [10, 15, 20, 25]:
        print('  celic z mito% >= {}%: {} ({:.1f}%)'.format(th, int((mp>=th).sum()), 100*(mp>=th).mean()))
    print('  -> malo celic nad 20% => mito filter malo spremeni; veliko => pomemben.')
else:
    print('  OPOZORILO: ni MT- genov (druga referenca?) -> mito filter ne bo delal, mito%=0.')

del _qc; gc.collect()


In [ ]:
adata_pbmc = load_h5_files(os.path.join(RAW_DIR, '*_GEX_sc_PBMC.h5'), meta_all)
print(f'PBMC RNA skupaj: {adata_pbmc.shape}')

# Sorted fajle izpustimo — so redundantni, metadata že loči CD4/CD8
# adata_pbmc_sorted = load_h5_files(os.path.join(RAW_DIR, '*_GEX_sc_sorted_*_PBMC.h5'), meta_all)

adata_tumor = load_h5_files(os.path.join(RAW_DIR, '*_GEX_sc_tumor.h5'), meta_all)
print(f'Tumor RNA skupaj: {adata_tumor.shape}')

adata_all = anndata.concat([adata_pbmc, adata_tumor], join='inner', index_unique=None)
print(f'Skupaj RNA (vse): {adata_all.shape}')

## 3. Poveži RNA barcodes z metadata

Vsaka celica v `.h5` fajlu ima barcode (npr. `AAACCTGAGAGACTAT-1`). Ta barcode se mora ujemati
z indeksom v metadata fajlu.

In [ ]:
## 3. Poveži RNA barcodes z metadata

meta_all['barcode_clean'] = meta_all.index.str.split('_').str[0]
valid_bc_patient = set(zip(meta_all['barcode_clean'], meta_all['Patient_ID']))

mask = pd.Series([
    (bc, pid) in valid_bc_patient
    for bc, pid in zip(adata_all.obs['barcode_clean'], adata_all.obs['Patient_ID'])
], index=adata_all.obs.index)

adata_filtered = adata_all[mask].copy()
print(f'RNA po filtriranju: {adata_filtered.shape}')

# Dedupliciraj meta pred joinom
meta_indexed = meta_all.set_index(['barcode_clean', 'Patient_ID'])
meta_indexed = meta_indexed[~meta_indexed.index.duplicated(keep='first')]

# Lookup prek composite ključa — brez joina po indeksu
keys = list(zip(adata_filtered.obs['barcode_clean'], adata_filtered.obs['Patient_ID']))
adata_filtered.obs['Stage']      = [meta_indexed.loc[k, 'Stage']      if k in meta_indexed.index else None for k in keys]
adata_filtered.obs['Tissue_str'] = [meta_indexed.loc[k, 'Tissue_str'] if k in meta_indexed.index else None for k in keys]
adata_filtered.obs['CellClass']  = [meta_indexed.loc[k, 'CellClass']  if k in meta_indexed.index else None for k in keys]

print(adata_filtered.obs[['Patient_ID', 'Stage', 'Tissue_str', 'CellClass']].head(3))
print(f'NaN v Stage: {adata_filtered.obs["Stage"].isna().sum()}')

In [ ]:
## 4. Sprosti RAM

import gc
for _var in ['adata_pbmc', 'adata_tumor', 'adata_all']:
    if _var in globals():
        del globals()[_var]
gc.collect()
print('RAM sproščen.')

## 5. Naloži TCR sekvence

Vsak `filtered_contig_*.csv.gz` vsebuje TCR sekvence po celicah (barcode).
Vzamemo samo beta verigo (TRB) — to je CDR3β ki ga TRIM uporablja.

In [ ]:
def load_tcr_files(pattern):
    """Naloži vse TCR contig fajle, vrni DataFrame z (barcode_clean, Patient_ID) → CDR3β, v_gene, j_gene."""
    files = sorted(glob.glob(pattern))
    print(f'Najdenih {len(files)} TCR fajlov')
    dfs = []
    for n, f in enumerate(files):
        patients = get_patients_from_filename(f)
        print(f"{n} --> {len(patients)} pacientov")

        if patients is None:
            print(f'  OPOZORILO: ne morem razbrati pacientov iz {os.path.basename(f)}, preskočim')
            continue

        df = pd.read_csv(f)
        df = df[(df['chain'] == 'TRB') &
                (df['productive'] == True) &
                (df['high_confidence'] == True) &
                (df['cdr3'].notna())]
        df['barcode_clean'] = df['barcode'].str.replace(r'-\d+$', '', regex=True)

        meta_sub = meta_all[meta_all['Patient_ID'].isin(patients)]
        bc_to_patient = meta_sub.groupby('barcode_clean')['Patient_ID'].first()
        df['Patient_ID'] = df['barcode_clean'].map(bc_to_patient)
        print(f"+ NA {len(df)}")
        df = df.dropna(subset=['Patient_ID'])
        print(f"- NA {len(df)}")

        dfs.append(df[['barcode_clean', 'Patient_ID', 'cdr3', 'v_gene', 'j_gene']])

    result = pd.concat(dfs, ignore_index=True)
    print(f"Before {len(result)}")
    result = result.drop_duplicates(['barcode_clean', 'Patient_ID'])
    print(f"After {len(result)}")
    return result

# PBMC TCR
tcr_pbmc = load_tcr_files(os.path.join(RAW_DIR, 'GSM*_filtered_contig_annotations_*_TCR_sc_PBMC.csv.gz'))

# Tumor TCR
tcr_tumor = load_tcr_files(os.path.join(RAW_DIR, 'GSM*_filtered_contig_annotations_*_TCR_sc_tumor.csv.gz'))

tcr_all = pd.concat([tcr_pbmc, tcr_tumor])
tcr_all = tcr_all.drop_duplicates(['barcode_clean', 'Patient_ID'])

print(f'TCR sekvenc skupaj: {len(tcr_all)}')
print(f'Unikatnih CDR3β sekvenc: {tcr_all["cdr3"].nunique()}')
print(tcr_all.head(3))

## 6. Sestavi data_labels DataFrame

Za vsako celico v RNA matriki določimo:
- `Tissue`: 0 = kri, 1 = tumor
- `Treatment Stage`: 0 = pred zdravljenjem, 1 = po zdravljenju
- `Patient`: celo število (0-indeksirano)
- `CDR3(Beta1)`: indeks TCR sekvence v `df_all_tcrs`

In [ ]:
obs = adata_filtered.obs.copy()

# Tissue: kri=0, tumor=1
obs['Tissue'] = (obs['Tissue_str'] == 'Tumor').astype(int)

# Treatment Stage: B1/Pre-Tx = 0 (pred), B2/Post-Tx = 1 (po)
pre_stages  = {'B1', 'Pre-Tx'}
post_stages = {'B2', 'Post-Tx'}
obs['Treatment Stage'] = obs['Stage'].apply(
    lambda s: 0 if s in pre_stages else (1 if s in post_stages else np.nan)
)

# Patient: string → integer (0-indeksiran)
patient_ids = sorted(obs['Patient_ID'].unique())
patient2idx = {p: i for i, p in enumerate(patient_ids)}
obs['Patient'] = obs['Patient_ID'].map(patient2idx)

print(f'Pacientov: {len(patient_ids)}')
print(f'Pacienti: {patient_ids}')
print(f'Celic z NaN Stage: {obs["Treatment Stage"].isna().sum()}')
print(obs[['Tissue', 'Treatment Stage', 'Patient']].value_counts().head(10))

In [ ]:
# Ustvari df_all_tcrs — DataFrame vseh unikatnih CDR3β sekvenc
unique_tcrs = tcr_all['cdr3'].unique()
df_all_tcrs = pd.DataFrame(index=unique_tcrs)

max_len = max(len(s) for s in unique_tcrs)
df_all_tcrs.index = [s.ljust(max_len) for s in df_all_tcrs.index]

tcr_seq2idx = {seq: i for i, seq in enumerate(df_all_tcrs.index)}

# CDR3(Beta1): indeks sekvence; celice brez TCR dobijo -1
tcr_lookup = tcr_all.set_index(['barcode_clean', 'Patient_ID'])['cdr3']
obs['CDR3_seq'] = [tcr_lookup.get((bc, pid)) for bc, pid in zip(obs['barcode_clean'], obs['Patient_ID'])]
obs['CDR3(Beta1)'] = obs['CDR3_seq'].apply(
    lambda s: tcr_seq2idx.get(s.ljust(max_len) if pd.notna(s) else None, -1)
)

# SubCellType: CD4=0, CD8=1
obs['SubCellType'] = (obs['CellClass'] == 'CD8').astype(int)

# V/J gene indices — globalni seznam vseh V in J genov
v_lookup = tcr_all.set_index(['barcode_clean', 'Patient_ID'])['v_gene'].fillna('')
j_lookup = tcr_all.set_index(['barcode_clean', 'Patient_ID'])['j_gene'].fillna('')

all_v_genes = sorted(v_lookup.unique().tolist())
all_j_genes = sorted(j_lookup.unique().tolist())
v2idx = {v: i for i, v in enumerate(all_v_genes)}
j2idx = {j: i for i, j in enumerate(all_j_genes)}

obs['v_gene'] = [v_lookup.get((bc, pid), '') for bc, pid in zip(obs['barcode_clean'], obs['Patient_ID'])]
obs['j_gene'] = [j_lookup.get((bc, pid), '') for bc, pid in zip(obs['barcode_clean'], obs['Patient_ID'])]
obs['tcr_v'] = obs['v_gene'].map(lambda v: v2idx.get(v if pd.notna(v) else '', 0))
obs['tcr_j'] = obs['j_gene'].map(lambda j: j2idx.get(j if pd.notna(j) else '', 0))

print(f'Unikatnih TCR sekvenc: {len(df_all_tcrs)}')
print(f'Max dolžina sekvence: {max_len}')
print(f'Celic z CDR3(Beta1) = -1 (brez TCR): {(obs["CDR3(Beta1)"] == -1).sum()}')
print(f'Celic z znano TCR sekvenco: {(obs["CDR3(Beta1)"] != -1).sum()} / {len(obs)}')
print(f'V genov: {len(all_v_genes)}, J genov: {len(all_j_genes)}')
print(f'SubCellType distribucija: {obs["SubCellType"].value_counts().to_dict()}')

In [ ]:
valid_mask = (obs['CDR3(Beta1)'] != -1) & (obs['Treatment Stage'].notna())
print(f'Celic pred filtriranjem: {len(obs)}')
print(f'Celic po filtriranju (z TCR in veljavnim Stage): {valid_mask.sum()}')

obs_final = obs[valid_mask].copy()

# Shrani GENE IMENA (var_names) — scFEA (flux pipeline) jih potrebuje (gene simboli).
# adata_filtered.var_names je poravnan s stolpci data_rna_final.
gene_names = adata_filtered.var_names.to_numpy()
with open(os.path.join(OUTPUT_DIR, 'gene_names.pkl'), 'wb') as f:
    pickle.dump(gene_names, f)
print(f'gene_names.pkl shranjen: {len(gene_names)} genov, primer: {list(gene_names[:5])}')

# ---------------------------------------------------------------------------
# NORMALIZACIJA RNA
# ---------------------------------------------------------------------------
# POPRAVEK (4.7.2026): prej je bila tu SAMO log1p(surovi counti). To NE ustreza
# avtorjevi normalizaciji. Original `data_processing.py:49-58` (library_size_normalize)
# naredi PER-CELICO:  log(x+eps) -> odstej per-cell min -> deli s per-cell max.
# Rezultat: izrazeni geni koncajo v ~[0.84, 1.0] (skoraj binarno "izrazen/ne"),
# nicle ostanejo 0. TRIM (VAE + PCA + lambda utezi + eval) je zasnovan za TA vhod;
# log1p (razpon 0..~11) da drugacen prostor -> nezvesta replikacija.
#
# Ker ima vsaka celica >>1 nizel gen (izrazi ~1768/36601), je per-cell min VEDNO
# log(eps). Zato transformacija OHRANI redkost: preslikajo se le nenicelne vrednosti.
from scipy.sparse import issparse, csr_matrix

X = csr_matrix(adata_filtered.X[valid_mask.values])   # SUROVI counti (celice x geni)

# (a) SUROVI counti — flux pipeline (scFEA) rabi surove counte, NE normaliziranih.
#     Flux notebook naj bere 'data_rna_counts.pkl' (NE vec expm1(data_rna)!).
data_rna_counts = X.copy()

# (b) Avtorjeva per-cell log + min-max normalizacija (sparse-ohranjajoca, vektorizirana)
eps = 1e-6
log_eps = np.log(eps)
row_max_counts = np.asarray(X.max(axis=1).todense()).ravel()   # najvecji count na celico
log_row_max = np.log(row_max_counts + eps)                      # = per-cell max od log(x+eps)
denom = log_row_max - log_eps                                   # per-cell (max - min)
denom[denom == 0] = 1.0                                         # varovalo (celica z 1 samim countom)

Xn = X.astype(np.float32).tocsr()
nnz_per_row = np.diff(Xn.indptr)
denom_per_nnz = np.repeat(denom, nnz_per_row).astype(np.float32)
Xn.data = ((np.log(Xn.data + eps) - log_eps) / denom_per_nnz).astype(np.float32)
data_rna_final = Xn

del adata_filtered
gc.collect()

print(f'data_rna_final oblika: {data_rna_final.shape}  (avtorjeva log+minmax normalizacija)')
print(f'  nenicelni razpon: min {data_rna_final.data.min():.3f}  max {data_rna_final.data.max():.3f}  (pricakovano ~0.8..1.0)')
print(f'data_rna_counts oblika: {data_rna_counts.shape}  (surovi counti za flux)')

## 8. Shrani `.pkl` fajle

In [ ]:
# data_rna.pkl (avtorjeva log+minmax normalizacija — za TRIM)
with open(os.path.join(OUTPUT_DIR, 'data_rna.pkl'), 'wb') as f:
    pickle.dump(data_rna_final, f)
print(f'data_rna.pkl shranjen: oblika {data_rna_final.shape}')

# data_rna_counts.pkl — SUROVI counti, LOCEN fajl za flux (scFEA). NE za TRIM.
# (Flux notebook naj bere tega, NE vec expm1(data_rna).)
with open(os.path.join(OUTPUT_DIR, 'data_rna_counts.pkl'), 'wb') as f:
    pickle.dump(data_rna_counts, f)
print(f'data_rna_counts.pkl shranjen (surovi counti za flux): oblika {data_rna_counts.shape}')

# data_labels.pkl — 8 stolpcev v enakem vrstnem redu kot original:
# 0:Tissue, 1:Treatment Stage, 2:SubCellType, 3:Patient, 4:CDR3(Beta1), 5:tcr_v, 6:tcr_j, 7:treatment
cols = ['Tissue', 'Treatment Stage', 'SubCellType', 'Patient', 'CDR3(Beta1)', 'tcr_v', 'tcr_j']
data_labels = obs_final[cols].copy()
data_labels['Treatment Stage'] = data_labels['Treatment Stage'].astype(int)
data_labels['treatment'] = 0  # v originalu nedefiniran — stolpec 7, nikjer se ne uporablja
with open(os.path.join(OUTPUT_DIR, 'data_labels.pkl'), 'wb') as f:
    pickle.dump(data_labels, f)
print(f'data_labels.pkl shranjen: oblika {data_labels.shape}')
print(f'Stolpci: {data_labels.columns.tolist()}')

# data_labels_str.pkl
cols_str = ['Tissue_str', 'Stage', 'Patient_ID', 'CDR3_seq']
data_labels_str = obs_final[cols_str].copy()
data_labels_str.columns = ['Tissue', 'Treatment Stage', 'Patient', 'CDR3(Beta1)']
with open(os.path.join(OUTPUT_DIR, 'data_labels_str.pkl'), 'wb') as f:
    pickle.dump(data_labels_str, f)
print(f'data_labels_str.pkl shranjen')

# df_all_tcrs.pkl — z 4 count stolpci (klonska velikost po kondiciji), kot original.
# Original (data_processing.py:115-135) sestavi stolpce '1'..'4' prek np.unique(return_counts)
# na vsaki kondiciji. Mi to ekvivalentno izpeljemo iz data_labels (CDR3(Beta1) x Tissue x Treatment Stage).
# KLJUC: counte poravnamo na OBSTOJECI vrstni red df_all_tcrs.index (NE sortiramo),
# ker CDR3(Beta1) indeksi v data_labels ze kazejo nanj.
n_tcrs = len(df_all_tcrs)
tcr_counts = np.zeros((n_tcrs, 4), dtype=np.int32)
# stolpci: 0=blood-pre(B1), 1=blood-post(B2), 2=tumor-pre, 3=tumor-post  -> imena '1','2','3','4' kot original
cond_map = {(0, 0): 0, (0, 1): 1, (1, 0): 2, (1, 1): 3}
for (t, s), ci in cond_map.items():
    sub = data_labels[(data_labels['Tissue'] == t) & (data_labels['Treatment Stage'] == s)]
    vc = sub['CDR3(Beta1)'].value_counts()
    tcr_counts[vc.index.values, ci] = vc.values
# vstavi kot stolpce; NaN namesto 0 za kondicije brez pojavitve (original ima NaN, eval dela .fillna(0))
df_all_tcrs_counts = df_all_tcrs.copy()
for ci, name in enumerate(['1', '2', '3', '4']):
    col = tcr_counts[:, ci].astype(float)
    col[col == 0] = np.nan
    df_all_tcrs_counts[name] = col
df_all_tcrs = df_all_tcrs_counts

with open(os.path.join(OUTPUT_DIR, 'df_all_tcrs.pkl'), 'wb') as f:
    pickle.dump(df_all_tcrs, f)
print(f'df_all_tcrs.pkl shranjen: {df_all_tcrs.shape} (index + 4 count stolpci)')
print(f'  Vsota countov po kondiciji [bpre,bpost,tpre,tpost]: {np.nan_to_num(df_all_tcrs.values).sum(axis=0).astype(int)}')
print(f'  Skupaj (mora biti = {len(data_labels)}): {int(np.nan_to_num(df_all_tcrs.values).sum())}')

## 9. Preverjanje

Preverimo da so fajli pravilni pred nadaljevanjem.

In [ ]:
# Naloži nazaj in preveri
with open(os.path.join(OUTPUT_DIR, 'data_rna.pkl'), 'rb') as f:
    check_rna = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'data_labels.pkl'), 'rb') as f:
    check_labels = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, 'df_all_tcrs.pkl'), 'rb') as f:
    check_tcrs = pickle.load(f)

print('=== PREVERJANJE ===')
print(f'data_rna:    {check_rna.shape}  (pričakovano: n_celic × n_genov)')
print(f'data_labels: {check_labels.shape}  (mora imeti enako vrstic kot data_rna, 8 stolpcev)')
print(f'df_all_tcrs: {len(check_tcrs)} sekvenc')
print()
print(f'Stolpci v data_labels: {check_labels.columns.tolist()}')
print(f'Vrednosti Tissue:          {sorted(check_labels["Tissue"].unique())}')
print(f'Vrednosti Treatment Stage: {sorted(check_labels["Treatment Stage"].unique())}')
print(f'Vrednosti SubCellType:     {sorted(check_labels["SubCellType"].unique())}  (0=CD4, 1=CD8)')
print(f'Pacientov:                 {check_labels["Patient"].nunique()}')
print(f'Max CDR3 indeks:           {check_labels["CDR3(Beta1)"].max()} (mora biti < {len(check_tcrs)})')

# Simuliraj col_* unpack kot ga delajo vsi downstream skripti
col_bloodtumor, col_prepost, col_celltype, col_patient, col_tcr, col_tcr_v, col_tcr_j, col_treatment = list(range(check_labels.shape[1]))
print(f'\ncol_bloodtumor={col_bloodtumor}, col_prepost={col_prepost}, col_celltype={col_celltype}, col_patient={col_patient}')
print(f'col_tcr={col_tcr}, col_tcr_v={col_tcr_v}, col_tcr_j={col_tcr_j}, col_treatment={col_treatment}')

# QC + normalizacija: brez praznih celic; nenicelne vrednosti v ~[0,1] (avtorjeva normalizacija)
from scipy.sparse import issparse
sums = np.asarray(check_rna.sum(axis=1)).ravel() if issparse(check_rna) else check_rna.sum(axis=1)
n_empty = int((sums == 0).sum())
nzmin = float(check_rna.data.min()) if issparse(check_rna) else float(check_rna[check_rna>0].min())
nzmax = float(check_rna.data.max()) if issparse(check_rna) else float(check_rna.max())
print(f'\n=== QC PREVERBA ===')
print(f'Praznih celic (vsota=0): {n_empty}  (mora biti 0 po QC min_genes=200)')
print(f'Nenicelni razpon data_rna: {nzmin:.3f} .. {nzmax:.3f}  (avtorjeva log+minmax -> ~0.8..1.0; NE log1p)')
print(f'Celic (mediana ne-praznih countov ni smiselna po normalizaciji — preveri v data_rna_counts.pkl)')

assert check_rna.shape[0] == check_labels.shape[0], 'NAPAKA: RNA in labels nimata enakega števila vrstic!'
assert check_labels['CDR3(Beta1)'].max() < len(check_tcrs), 'NAPAKA: CDR3 indeks je izven obsega!'
assert check_labels.shape[1] == 8, f'NAPAKA: data_labels mora imeti 8 stolpcev, ima {check_labels.shape[1]}!'
assert n_empty == 0, f'NAPAKA: {n_empty} praznih celic!'
assert nzmax <= 1.0 + 1e-4, f'NAPAKA: data_rna ni normaliziran v [0,1] (max={nzmax:.3f}) — je se log1p?'
print()
print('Vse preverjeno — preprocessing uspešen!')